# Chapter 19 — Correlation, Covariance & Feature Selection
*Track 3: Data Scientists*

## 🎯 Learning Objectives
- Distinguish correlation from covariance
- Identify multicollinearity and its effects on models
- Apply correlation-based feature selection

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
import pandas as pd
import seaborn as sns
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression

rng = np.random.default_rng(42)
plt.style.use("seaborn-v0_8-whitegrid")
print("Libraries loaded ✅")

## 1. Concept Review — Covariance and Correlation

$$\text{Cov}(X,Y) = E[(X-\mu_X)(Y-\mu_Y)]$$

$$r = \text{Cor}(X,Y) = \frac{\text{Cov}(X,Y)}{\sigma_X \sigma_Y} \in [-1, 1]$$

- Covariance: scale-dependent; hard to interpret
- Pearson r: scale-free; measures **linear** relationship
- Spearman ρ: rank-based; measures **monotonic** relationship
- Kendall τ: concordance pairs; robust to outliers

In [ ]:
# Visualise different correlation patterns
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
scenarios = [
    ("Strong positive", 0.95, "linear"),
    ("Moderate positive", 0.6, "linear"),
    ("No correlation", 0.0, "linear"),
    ("Strong negative", -0.9, "linear"),
    ("Non-linear", 0.0, "quadratic"),
    ("Outlier effect", 0.3, "outlier"),
    ("Anscombe Quartet", 0, "anscombe"),
    ("Spearman vs Pearson", 0, "monotone"),
]
n = 100
for ax, (label, r, stype) in zip(axes.flat, scenarios):
    if stype == "linear":
        x = rng.normal(0, 1, n)
        y = r*x + np.sqrt(1-r**2)*rng.normal(0, 1, n)
        pearson = np.corrcoef(x, y)[0,1]
        ax.scatter(x, y, alpha=0.5, s=20)
        ax.set_title(f"{label}
r={pearson:.2f}", fontsize=9)
    elif stype == "quadratic":
        x = rng.uniform(-2, 2, n)
        y = x**2 + rng.normal(0, 0.5, n)
        pearson = np.corrcoef(x, y)[0,1]
        spearman = stats.spearmanr(x, y).statistic
        ax.scatter(x, y, alpha=0.5, s=20)
        ax.set_title(f"Quadratic
Pearson={pearson:.2f}, Spearman={spearman:.2f}", fontsize=9)
    elif stype == "outlier":
        x = rng.normal(0, 1, n)
        y = 0.1*x + rng.normal(0, 1, n)
        x = np.append(x, [10]); y = np.append(y, [10])
        pearson = np.corrcoef(x, y)[0,1]
        ax.scatter(x, y, alpha=0.5, s=20)
        ax.set_title(f"Outlier distortion
r={pearson:.2f}", fontsize=9)
    elif stype == "anscombe":
        xA = np.array([10,8,13,9,11,14,6,4,12,7,5])
        yA = np.array([8.04,6.95,7.58,8.81,8.33,9.96,7.24,4.26,10.84,4.82,5.68])
        pearson = np.corrcoef(xA, yA)[0,1]
        ax.scatter(xA, yA, s=30)
        ax.set_title(f"Anscombe's Quartet
r={pearson:.2f}", fontsize=9)
    else:
        x = rng.uniform(0, 5, n)
        y = np.exp(x/2) + rng.normal(0, 0.5, n)
        pearson = np.corrcoef(x, y)[0,1]
        spearman = stats.spearmanr(x, y).statistic
        ax.scatter(x, y, alpha=0.5, s=20)
        ax.set_title(f"Monotone nonlinear
Pearson={pearson:.2f}, Spearman={spearman:.2f}", fontsize=9)
plt.suptitle("Correlation Patterns", fontweight="bold", fontsize=13)
plt.tight_layout(); plt.show()

## 2–6. Multicollinearity, VIF, Feature Selection, Practice

In [ ]:
# Variance Inflation Factor (VIF) for multicollinearity
from sklearn.preprocessing import StandardScaler

diabetes = load_diabetes()
X = pd.DataFrame(diabetes.data, columns=diabetes.feature_names)

def compute_vif(X_df):
    vif = {}
    for col in X_df.columns:
        other_cols = [c for c in X_df.columns if c != col]
        r2 = LinearRegression().fit(X_df[other_cols], X_df[col])
        r2_val = r2.score(X_df[other_cols], X_df[col])
        vif[col] = 1 / (1 - r2_val) if r2_val < 1 else np.inf
    return pd.Series(vif).sort_values(ascending=False)

vif_scores = compute_vif(X)
print("VIF Scores (>5 suggests multicollinearity):")
print(vif_scores.round(2))

vif_scores.plot(kind="barh")
plt.axvline(5, color="red", linestyle="--", label="VIF=5 threshold")
plt.xlabel("VIF"); plt.title("Variance Inflation Factors — Diabetes Dataset")
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Correlation heatmap and feature selection
corr_matrix = X.corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("Feature Correlation Matrix — Diabetes Dataset")
plt.tight_layout(); plt.show()

# Select features with |r| > 0.5 with target
y_d = diabetes.target
target_corr = pd.Series({col: abs(np.corrcoef(X[col], y_d)[0,1]) for col in X.columns})
selected = target_corr[target_corr > 0.2].index.tolist()
print(f"
Features with |r|>0.2 with target: {selected}")

In [ ]:
# Practice: Pearson vs Spearman on income data
income = rng.lognormal(10, 1, 200)
age = rng.uniform(22, 65, 200)
income += 500 * (age - 22)  # monotone but nonlinear relationship

pearson_r, p_pearson = stats.pearsonr(age, income)
spearman_r, p_spearman = stats.spearmanr(age, income)
print(f"Pearson r={pearson_r:.4f}, p={p_pearson:.4f}")
print(f"Spearman ρ={spearman_r:.4f}, p={p_spearman:.4f}")
print("Spearman catches the monotone relationship better for skewed data.")